# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, review, and analyze the FAIR² dataset—clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors—using the `mlcroissant` library.

### Dataset Source
The dataset is provided in FAIR Croissant format at the following URL:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their IDs (`@id`s), and structure.


In [ ]:
# List all record sets and their fields, referencing by `@id`
from collections import defaultdict

def get_record_sets(ds):
    # Return list of RecordSet metadata as croissant objects
    record_sets = []
    for obj in ds.metadata.fields:
        if getattr(obj, '@type', None) == 'cr:RecordSet':
            record_sets.append(obj)
    if hasattr(ds.metadata, 'recordSet') and ds.metadata.recordSet:
        # If record sets are referenced from the root metadata
        for rs in ds.metadata.recordSet:
            if hasattr(rs, '@id'):
                record_sets.append(rs)
    return record_sets

# In croissant 1.2+ most datasets provide dataset.record_sets to enumerate them directly
try:
    record_sets = list(dataset.record_sets())  # Returns list of croissant.RecordSet
except AttributeError:
    # Fallback
    record_sets = []

if not record_sets:
    # Try heuristics if no built-in listing
    record_sets = []
    for r in getattr(metadata, 'recordSet', []):
        record_sets.append(r)

# Print record sets by @id, and each field by @id
for rs in record_sets:
    rs_id = getattr(rs, '@id', rs) if isinstance(rs, dict) or hasattr(rs, '__dict__') else str(rs)
    try:
        rs_name = getattr(rs, 'name', None)
    except Exception:
        rs_name = None
    print(f"RecordSet @id: {rs_id}")
    if rs_name:
        print(f"  Name: {rs_name}")
    # Try to get fields/columns for each record set
    if hasattr(rs, 'field') and rs.field:
        print("  Fields (by @id):")
        for field in rs.field:
            field_id = getattr(field, '@id', 'UNKNOWN-FIELD-ID')
            field_name = getattr(field, 'name', '')
            print(f"    - {field_id}   Name: {field_name}")
    elif hasattr(rs, 'column') and rs.column:
        print("  Columns (by @id):")
        for column in rs.column:
            col_id = getattr(column, '@id', 'UNKNOWN-COLUMN-ID')
            col_name = getattr(column, 'name', '')
            print(f"    - {col_id}   Name: {col_name}")
    print("")

# For datasets with one main RecordSet (i.e., tabular), print a sample of records and available keys by @id
# If record_sets is empty, try to load default records:
if not record_sets:
    print('\nNo explicit record sets found. Attempting to show default records (if available):')
    try:
        sample = next(dataset.records())
        print('Keys in sample record:', sample.keys())
    except Exception as e:
        print('No record preview available:', e)

## 3. Data Extraction
Load data from each record set (referenced by `@id`) into Pandas DataFrames for downstream analysis. (You can select the most relevant RecordSet for your task.)

In [ ]:
# We'll attempt to automatically discover tabular record sets or simply load the default/first one.
import warnings
warnings.filterwarnings('ignore')

# If record_sets is empty, try default records
dataframes = dict()
record_set_ids = []

def get_main_record_set(ds):
    # Try dataset.record_sets()
    try:
        for rs in ds.record_sets():
            return rs
    except Exception:
        pass
    # Fall back to metadata.recordSet
    if hasattr(ds.metadata, 'recordSet'):
        rs_list = ds.metadata.recordSet
        if isinstance(rs_list, list) and rs_list:
            return rs_list[0]
    return None

# Get main RecordSet
main_rs = get_main_record_set(dataset)
if main_rs:
    main_rs_id = getattr(main_rs, '@id', str(main_rs))
    record_set_ids.append(main_rs_id)
else:
    # Try first records() if no explicit record set
    main_rs_id = None

# Load DataFrames per RecordSet
if record_set_ids:
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"RecordSet @id: {rs_id}, n_records: {len(df)}")
        print(f"Columns (@id): {df.columns.tolist()}")
        display(df.head())
else:
    # Fallback to whole-dataset records
    records = list(dataset.records())
    df = pd.DataFrame(records)
    dataframes['__default__'] = df
    print(f"Default record set n_records: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
We'll perform some common analysis steps on a numeric field, referencing it by its schema `@id` as per the Croissant definition.

**Note:** Modify below to target the most analytic-relevant field (e.g., "Age" or "Interval_between_cancers"). You may view available columns from above output.

In [ ]:
# --- Analysis by column @id (edit as needed per dataset schema) ---
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Use main record set loaded above
record_set_id = record_set_ids[0] if record_set_ids else '__default__'
df = dataframes[record_set_id]

print(f'Columns: {df.columns.tolist()}')

# Let's select a numeric field by @id (assume typical medical fields such as 'Age'/'Interval_between_diagnoses')
# Modify below to match your available column names or @id values from data overview

candidate_numeric_ids = [col for col in df.columns if any(key in col.lower() for key in ['age', 'interval', 'duration', 'years', 'months', 'number', 'count'])]
if candidate_numeric_ids:
    numeric_field_id = candidate_numeric_ids[0]
    print(f"Selected numeric field for analysis: {numeric_field_id}")
else:
    numeric_field_id = df.select_dtypes(include=np.number).columns[0] if len(df.select_dtypes(include=np.number).columns) else df.columns[0]
    print(f"Fallback numeric field: {numeric_field_id}")

# Clean missing/parse as numeric if needed
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
df_nonan = df.dropna(subset=[numeric_field_id])

threshold = df_nonan[numeric_field_id].mean()
filtered_df = df_nonan[df_nonan[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another attribute (e.g., sex, cancer_type, msi_status)
candidate_group_fields = [col for col in df.columns if any(gk in col.lower() for gk in ['sex', 'gender', 'msi', 'status', 'cancer', 'group', 'type', 'anatomical'])]
group_field_id = candidate_group_fields[0] if candidate_group_fields else None
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with one categorical attribute (if present).


In [ ]:
# Numeric Field Distribution
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

# Boxplot by group if group_field_id is available and valid
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    order = df[group_field_id].dropna().unique()
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, order=order)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, filtering, normalizing, and grouping data from the FAIR² dataset using the `mlcroissant` library. Using Croissant schema `@id` references ensures that data extraction and manipulation are both robust and reproducible.

- Loaded data directly from a remote Croissant schema.
- Inspected available record sets and fields using their `@id`.
- Extracted records into DataFrames, filtered records, normalized and explored numeric fields, and visualized key variables.

For additional exploration, you may adjust the column `@id`s and aggregations to analyze clinical subgroups or molecular features within the dataset.